# Leçon 11 - Protocole Agent-à-Agent (A2A)


## Configuration


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Qu'est-ce que le protocole A2A ?

Le **protocole Agent-à-Agent (A2A)** est une norme ouverte qui permet aux agents d'IA de communiquer,
se découvrir et de collaborer — même lorsqu'ils sont construits sur des frameworks différents ou hébergés
par des services différents.

Key concepts:

- **Découverte** – Les agents publient une *Carte d'agent* qui décrit leurs capacités, ce qui
  permet aux autres agents (ou orchestrateurs) de trouver le spécialiste approprié pour une tâche.
- **Transmission de messages** – Les agents échangent des messages structurés via un protocole commun, de sorte qu'une
  demande d'un agent puisse être comprise et exécutée par un autre, indépendamment de son
  implémentation.
- **Cycle de vie des tâches** – A2A définit des états tels que *submitted*, *working*, *completed*, et
  *failed*, donnant à l'orchestrateur une visibilité complète sur la progression d'une tâche déléguée.

Dans cette leçon, nous simulons une collaboration de type A2A en branchant trois agents de voyage spécialisés
dans un flux de travail où chaque agent apporte son expertise et transmet les résultats au suivant.


## Création d'agents de voyage spécialisés


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Collaboration multi-agent via un flux de travail

Nous connectons les trois agents dans un flux de travail séquentiel qui reflète le passage de messages A2A:

1. **CurrencyExchangeAgent** reçoit la demande de l'utilisateur et fournit des conseils sur les devises.
2. **ActivityPlannerAgent** reçoit le contexte enrichi et ajoute des recommandations d'activités.
3. **TravelManagerAgent** synthétise les deux entrées en un résumé de voyage final.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Comprendre A2A en production

Dans un environnement de production, le protocole A2A permet des scénarios inter-services puissants :

| Capability | Description |
|---|---|
| **Cross-framework interop** | Un agent développé avec un framework peut déléguer des tâches à un agent développé avec n'importe quel autre framework compatible A2A, permettant une véritable interopérabilité entre organisations. |
| **Service boundaries** | Les agents peuvent résider dans des microservices séparés, des régions cloud ou même dans des organisations différentes tout en collaborant de manière transparente. |
| **Dynamic discovery** | Un orchestrateur peut interroger un registre Agent Card à l'exécution pour trouver le spécialiste le mieux adapté à une sous-tâche donnée. |
| **Streaming & push notifications** | A2A prend en charge les événements envoyés par le serveur (SSE) pour les mises à jour d'avancement en temps réel et les notifications push pour les tâches de longue durée. |

Le flux de travail que nous avons construit ci-dessus est une version simplifiée, exécutée dans le même processus, de ce modèle. Dans un déploiement réel, chaque agent exposerait un point de terminaison HTTP, publierait une Agent Card, et communiquerait via le protocole A2A JSON-RPC.


## Résumé

Dans cette leçon vous avez appris:

1. **Ce qu'est le protocole A2A** — une norme ouverte pour la découverte entre agents, la messagerie,
   et la gestion des tâches.
2. **Comment créer des agents spécialisés** — un agent Currency Exchange, un agent Activity Planner,
   et un orchestrateur Travel Manager.
3. **Comment relier les agents dans un workflow** — en utilisant `WorkflowBuilder` pour modéliser le
   passage séquentiel de messages entre les agents.
4. **Comment A2A fonctionne en production** — permettant la collaboration entre frameworks et services
   avec découverte dynamique et mises à jour en flux continu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Clause de non-responsabilité :
Ce document a été traduit à l'aide du service de traduction par IA Co-op Translator (https://github.com/Azure/co-op-translator). Bien que nous nous efforcions d'en assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original, dans sa langue d'origine, doit être considéré comme la source faisant foi. Pour les informations critiques, une traduction professionnelle réalisée par un humain est recommandée. Nous déclinons toute responsabilité en cas de malentendus ou de mauvaises interprétations résultant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
